# Cars 4 You Machine Learning Project


## Group Members

- Diogo Carvalho - 20221935
- Luiza Salum - 20221902
- Ricardo Pereira - 20250343


In [ ]:
from __future__ import annotations

import warnings
from typing import TYPE_CHECKING, cast

import optuna
import polars as pl
from optuna.integration import OptunaSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

from funcs.pipeline import build_feature_selection_pipeline

%load_ext autoreload
%autoreload 2

from funcs.data_import import describe_data, import_data
from funcs.pipeline import build_preprocessing_pipeline

if TYPE_CHECKING:
    from typing import Any

    from numpy.typing import NDArray
    from pandas import DataFrame, Series

warnings.filterwarnings("ignore", category=FutureWarning)

pl.Config.set_tbl_cols(-1)

## Importing Data


In [ ]:
train_df: pl.DataFrame = import_data("data/train.csv")
test_df: pl.DataFrame = import_data("data/test.csv")

## Exploration


In [ ]:
describe_data(
    train_df,
    [
        "price",
        "mileage",
        "tax",
        "mpg",
        "engineSize",
        "paintQuality%",
        "previousOwners",
    ],
    ["Brand", "model", "year", "transmission", "fuelType", "hasDamage"],
)

Our training dataset contains:

- No duplicate values
- Missing values in all columns except carID (the index) and price (the target
  variable).
- Outliers
- Some data types or values that do not make sense such as:
  - Floats in the year, mileage, tax, paintQuality%, previousOwners and
    hasDamage columns, these should be integers.
  - Negative values in mileage, tax, mpg, engineSize and previousOwners, these
    should always be positive.
  - Unrealistic values such as engine size or miles per gallon being 0.


In [ ]:
x_train: pl.DataFrame = train_df.drop("price", "paintQuality%")
y_train: pl.DataFrame = train_df.select("price")

## Pipeline


### Preprocessing

#### Inconsistencies and Outliers

As mentioned before, our dataset contains outliers and inconsistencies. To
tackle the unrealistic values we decided to bind the data to values that made
sense.

The values that we decided on are as follows:

- Mileage:
  - Minimum: 1, since Cars 4 You is a resale company, the cars have to be used.
  - Maximum: None, there should be no upper limit to the mileage in a car, if
    anything, this should affect the quality of the car and, consequently, its
    price.
- Tax:
  - Minimum: 0, tax exemptions
  - Maximum: 400, values of tax above 360-370 are outliers, the next value found
    is above 500, 400 was chosen due to it being a nice, round number. Even
    though the outliers don't seem to be unrealistic, we were worried that
    leaving them in would negatively impact the performance of our model, as
    such, this upper breakpoint is subject to change.
- Miles per Gallon:
  - Minimum: 20, after doing some research, this seemed like a reasonable
    minimum.
  - Maximum: 200, same reasoning as above.
- Engine Size:
  - Minimum: 1.0, same reasoning as miles per gallon
  - Maximum: 6.0, same reasoning as miles per gallon
- Paint Quality %:
  - Minimum: 0, it's a percentage value, there should not be values under 0
  - Maximum: 100, same reasoning as above, there should not be values above 100
- Previous Owners:
  - Minimum: 0, a car cannot have a negative amount of previous owners
  - Maximum: None, considered a maximum of 4, due to there being outliers above
    this value. However, it's not a nonsensical value, so we left it for now,
    just like tax, this is subject to change.
- Year:
  - Minimum: 0.
  - Maximum: 2020, the data is from 2020, as such, year cannot be higher than
    2020, we don't think Cars 4 You is in possession of a time machine.

#### Missing Values

##### Metric Features

Filling with the median of the respective feature. There was no real reasoning
in this decision, as such, it is very subject to change once a more careful
analysis is done.

##### Boolean Features

Filling with 0. The only boolean feature present in the dataset is the hasDamage
feature. We can assume that if a car was checked by a mechanic, and was damaged,
it would not have been forgotten. As such filling with 0 did not seem
unreasonable. Even without this assumption, every non-missing value is 0, and,
with only 2.04% of missing values, we can safely assume that these missing
values are much more likely to be 0 than 1.

##### Categorical Features

We're merging the null and the unknown columns after dummies are created, e.g.
`brand_null` and `brand_unknown` get merged. If the null dummy column exists for
a categorical feature but the unknown dummy column doesn't, the former gets
renamed to `feature_name_unknown` where `feature_name` is the name of the
categorical feature. This would be equivalent to filling with "`unknown`".

#### Issues in Categorical Features

The columns for the brand, model, transmission and fuel type of the car present
spelling mistakes. Fixing brand, transmission and fuel type is easy, as the
correct values are easy to discover. We just have to compare each of the values
in these columns to the correct values and replace by the appropriate, corrected
value.

The model column however, was harder to fix. There is a lot of overlap between
models of different brands, for example the i3 from BMW and the I30 from
Hyundai, if we are presented with a model i3, should we leave it as i3, or
should we replace it by i30? Just getting the list of corrected models was not
enough, we had to look for a list of models for each of the brands present in
the dataset. After acquiring this list, we fixed the values by comparing the
current value to the ones in the list, taking into account the brand of the
entry (if it is not missing), this way, a i3 entry with brand BMW will remain as
i3 whilst a i3 entry with brand Hyundai will be replaced by i30.

There's also the situation on which we want to fix a model, but don't have the
brand of an entry, in this case, we first got the brand of the entry by
comparing its model to the aforementioned list of models, getting the brand that
the model belongs to, then running the function to fix models once more.

Thus the process is as follows:

- Run `fix models` to fix all models possible
- Run `fix models with no brand`
- Run `fix models` again to fix more models, now that brands is filled

For this process to run as smoothly as possible, some assumptions had to be
made, they are as follows:

1. If brand is missing, models `i3` and `i8` belong to BMW. This was concluded
   by manually looking into the dataset, and comparing the remaining features.
2. No more than 2 characters were removed in models. This avoids some cases of
   multiple matches, for example, if there was an entry `cl`, it will be
   replaced by `clk` and not `cl class`, `cla class` or `clc class` due to these
   being much longer than the entry present in the dataset.
3. For models that have exactly 2 matches we decided to maintain the value
   present in the dataset instead of trying to replace it. This would lead to an
   entry having multiple matches, which could only be resolvable by comparing
   the remaining features of the entry to those of the possible matches. We
   decided against this approach, simply because keeping the original entry was
   easier. An example of this are the models `ka` and `ka+` and `mokka` and
   `mokka x`.
4. Model entries `a` and `q` belong to Audi. Model entries `x` belong to BMW.
   Any model that starts with these letters besides the ones that belong to the
   aforementioned brands have a character difference larger than 2, as such,
   they're invalidated by assumption 2.
5. Model entries `k` are considered `ka` and not `ka+`. The features of these
   models are extremely similar, therefore, whether we assign `k` to `ka` or
   `ka+` shouldn't really matter.

#### Dummy Variables

Currently we are making n dummies for each categorical feature instead of n-1, n
being the number of different values existent in that feature. This is done to
ensure that all columns of the training set appear in the test set, we are aware
this introduces multicollinearity.

### Data Scaling

We decided to go with normalization.

As we're using polars, we wanted to make use of our own methods as much as
possible to avoid excessive conversion to pandas dataframes/series or numpy
arrays. Normalization was the one that came up to discussion first and as such
we decided to implement it.


In [ ]:
preproc_pipeline: Pipeline = build_preprocessing_pipeline(
    metric_features=[
        "mileage",
        "tax",
        "mpg",
        "engineSize",
        "previousOwners",
    ],
    bool_features=[
        "hasDamage",
    ],
    categorical_features=[
        "Brand",
        "model",
        "year",
        "transmission",
        "fuelType",
    ],
    thresholds={
        "mileage": {"lower": 1, "upper": None},
        "tax": {"lower": 0, "upper": 400},
        "mpg": {"lower": 20, "upper": 200},
        "engineSize": {"lower": 1.0, "upper": 6.0},
        "previousOwners": {"lower": 0, "upper": None},
        "year": {"lower": 0, "upper": 2020},
    },
    winsorize=False,
    unneeded_float_features=[
        "year",
        "mileage",
        "tax",
        "previousOwners",
        "hasDamage",
    ],
    scaling_exclude_selector=("_", "hasDamage", "carID"),
    columns_to_coalesce=(
        "Brand",
        "model",
        "transmission",
        "fuelType",
        "year",
    ),
)

### Feature Selection

Our feature selection goes as follows:

- Removing paintQuality% as it is a feature whose values are given by the
  mechanic. Since we want to create a model capable of evaluating the price of a
  car based on the user’s input without needing the car to be taken to a
  mechanic, this feature is irrelevant.
- Remove features that are constant
- Remove features with high correlation (>0.8)
- Select the most important features based on their
  [importance to the model](https://scikit-learn.org/stable/modules/feature_selection.html#select-from-model)
  using
  [SelectFromModel](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html)
  (with the exception of KNN as it does not work with SelectFromModel, for KNN
  we performed this step using a Random Forest estimator).


In [ ]:
fs_pipeline_lr: Pipeline = build_feature_selection_pipeline(
    threshold=0.8,
    estimator=LinearRegression(),
)
# Random Forest is used on purpose as KNeighborsRegressor does not work with SelectFromModel
fs_pipeline_knn: Pipeline = build_feature_selection_pipeline(
    threshold=0.8,
    estimator=RandomForestRegressor(max_depth=10, n_estimators=300),
)
fs_pipeline_dt: Pipeline = build_feature_selection_pipeline(
    threshold=0.8,
    estimator=DecisionTreeRegressor(max_depth=10),
)
fs_pipeline_rf: Pipeline = build_feature_selection_pipeline(
    threshold=0.8,
    estimator=RandomForestRegressor(max_depth=10, n_estimators=300),
)
fs_pipeline_en: Pipeline = build_feature_selection_pipeline(
    threshold=0.8,
    estimator=ElasticNet(
        random_state=42,
    ),
)

### Model

We decided to choose the following estimators:

- [Linear Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).
- [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [Decision Tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- [KNN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)
- [Elastic Net](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)

#### Model Selection/Tuning

We are performing 5 fold cross validation for all models. At the same time we
are tuning the hyperparameters for each model (except Linear Regression) using
Optuna.


In [ ]:
lr_pipeline = Pipeline(
    steps=[
        ("preprocessing", preproc_pipeline),
        ("feature_selection", fs_pipeline_lr),
        ("regressor", LinearRegression()),
    ]
)

cross_val_lr = cross_validate(
    lr_pipeline,
    cast("DataFrame", x_train),
    cast("Series", y_train),
    cv=5,
    scoring=(
        "r2",
        "neg_mean_absolute_error",
        "neg_mean_squared_error",
        "neg_root_mean_squared_error",
    ),
    n_jobs=-1,
)

lr_pipeline.fit(cast("DataFrame", x_train), cast("Series", y_train))

lr_predictions: NDArray[Any] = cast(
    "NDArray[Any]", lr_pipeline.predict(cast("DataFrame", test_df))
).flatten()

pl.DataFrame(
    {"carID": test_df.get_column("carID"), "price": lr_predictions}
).write_csv("data/submission-lr.csv")

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessing", preproc_pipeline),
        ("feature_selection", fs_pipeline_rf),
        (
            "regressor",
            RandomForestRegressor(criterion="squared_error", random_state=42),
        ),
    ]
)

optuna_search_rf = OptunaSearchCV(
    rf_pipeline,
    {
        "regressor__n_estimators": optuna.distributions.IntDistribution(
            10, 300
        ),
        "regressor__max_depth": optuna.distributions.IntDistribution(2, 32),
        "regressor__min_samples_split": optuna.distributions.IntDistribution(
            2, 10
        ),
        "regressor__min_samples_leaf": optuna.distributions.IntDistribution(
            1, 10
        ),
        "regressor__max_features": optuna.distributions.FloatDistribution(
            0.0, 1.0
        ),
    },
    cv=5,
    n_jobs=-1,
    n_trials=100,
    random_state=42,
    refit=True,
    scoring="neg_root_mean_squared_error",
)

optuna_search_rf.fit(cast("DataFrame", x_train), cast("Series", y_train))

rf_predictions: NDArray[Any] = cast(
    "NDArray[Any]", optuna_search_rf.predict(cast("DataFrame", test_df))
).flatten()

pl.DataFrame(
    {"carID": test_df.get_column("carID"), "price": rf_predictions}
).write_csv("data/submission-rf.csv")

In [ ]:
dt_pipeline = Pipeline(
    steps=[
        ("preprocessing", preproc_pipeline),
        ("feature_selection", fs_pipeline_dt),
        ("regressor", DecisionTreeRegressor(random_state=42)),
    ]
)

optuna_search_dt = OptunaSearchCV(
    dt_pipeline,
    {
        "regressor__max_depth": optuna.distributions.IntDistribution(2, 32),
        "regressor__min_samples_split": optuna.distributions.IntDistribution(
            2, 10
        ),
        "regressor__min_samples_leaf": optuna.distributions.IntDistribution(
            1, 10
        ),
    },
    cv=5,
    n_jobs=-1,
    n_trials=100,
    random_state=42,
    refit=True,
    scoring="neg_root_mean_squared_error",
)

optuna_search_dt.fit(cast("DataFrame", x_train), cast("Series", y_train))

dt_predictions: NDArray[Any] = cast(
    "NDArray[Any]", optuna_search_dt.predict(cast("DataFrame", test_df))
).flatten()

pl.DataFrame(
    {"carID": test_df.get_column("carID"), "price": dt_predictions}
).write_csv("data/submission-dt.csv")

In [ ]:
knn_pipeline = Pipeline(
    steps=[
        ("preprocessing", preproc_pipeline),
        ("feature_selection", fs_pipeline_knn),
        ("regressor", KNeighborsRegressor()),
    ]
)

optuna_search_knn = OptunaSearchCV(
    knn_pipeline,
    {
        "regressor__n_neighbors": optuna.distributions.IntDistribution(2, 50),
        "regressor__weights": optuna.distributions.CategoricalDistribution(
            ["uniform", "distance"]
        ),
        "regressor__algorithm": optuna.distributions.CategoricalDistribution(
            ["ball_tree", "kd_tree"]
        ),
        "regressor__p": optuna.distributions.IntDistribution(1, 2),
    },
    cv=5,
    n_jobs=-1,
    n_trials=100,
    random_state=42,
    refit=True,
    scoring="neg_root_mean_squared_error",
)

optuna_search_knn.fit(cast("DataFrame", x_train), cast("Series", y_train))

knn_predictions: NDArray[Any] = cast(
    "NDArray[Any]", optuna_search_knn.predict(cast("DataFrame", test_df))
).flatten()

pl.DataFrame(
    {"carID": test_df.get_column("carID"), "price": knn_predictions}
).write_csv("data/submission-knn.csv")

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessing", preproc_pipeline),
        ("feature_selection", fs_pipeline_en),
        (
            "regressor",
            ElasticNet(random_state=42),
        ),
    ]
)

optuna_search_en = OptunaSearchCV(
    pipeline,
    {
        "regressor__alpha": optuna.distributions.FloatDistribution(0.01, 10),
        "regressor__l1_ratio": optuna.distributions.FloatDistribution(0.1, 1),
        "regressor__tol": optuna.distributions.FloatDistribution(1e-4, 0.1),
    },
    cv=5,
    n_jobs=-1,
    n_trials=200,
    random_state=42,
    refit=True,
    scoring="neg_root_mean_squared_error",
)

optuna_search_en.fit(cast("DataFrame", x_train), cast("Series", y_train))

en_predictions: NDArray[Any] = cast(
    "NDArray[Any]", optuna_search_en.predict(cast("DataFrame", test_df))
).flatten()

pl.DataFrame(
    {"carID": test_df.get_column("carID"), "price": en_predictions}
).write_csv("data/submission-en.csv")

In [ ]:
cross_val_rf = cross_validate(
    optuna_search_rf.best_estimator_,
    cast("DataFrame", x_train),
    cast("Series", y_train),
    cv=5,
    scoring=(
        "r2",
        "neg_mean_absolute_error",
        "neg_mean_squared_error",
        "neg_root_mean_squared_error",
    ),
    n_jobs=-1,
)

cross_val_dt = cross_validate(
    optuna_search_dt.best_estimator_,
    cast("DataFrame", x_train),
    cast("Series", y_train),
    cv=5,
    scoring=(
        "r2",
        "neg_mean_absolute_error",
        "neg_mean_squared_error",
        "neg_root_mean_squared_error",
    ),
    n_jobs=-1,
)

cross_val_knn = cross_validate(
    optuna_search_knn.best_estimator_,
    cast("DataFrame", x_train),
    cast("Series", y_train),
    cv=5,
    scoring=(
        "r2",
        "neg_mean_absolute_error",
        "neg_mean_squared_error",
        "neg_root_mean_squared_error",
    ),
    n_jobs=-1,
)

cross_val_en = cross_validate(
    optuna_search_en.best_estimator_,
    cast("DataFrame", x_train),
    cast("Series", y_train),
    cv=5,
    scoring=(
        "r2",
        "neg_mean_absolute_error",
        "neg_mean_squared_error",
        "neg_root_mean_squared_error",
    ),
    n_jobs=-1,
)

In [ ]:
scores = pl.DataFrame(
    {
        "estimator": (
            "LinearRegression",
            "RandomForest",
            "DecisionTree",
            "KNeighbors",
            "ElasticNet",
        ),
        "R2": (
            cross_val_lr["test_r2"].mean(),
            cross_val_rf["test_r2"].mean(),
            cross_val_dt["test_r2"].mean(),
            cross_val_knn["test_r2"].mean(),
            cross_val_en["test_r2"].mean(),
        ),
        "MAE": (
            -cross_val_lr["test_neg_mean_absolute_error"].mean(),
            -cross_val_rf["test_neg_mean_absolute_error"].mean(),
            -cross_val_dt["test_neg_mean_absolute_error"].mean(),
            -cross_val_knn["test_neg_mean_absolute_error"].mean(),
            -cross_val_en["test_neg_mean_absolute_error"].mean(),
        ),
        "MSE": (
            -cross_val_lr["test_neg_mean_squared_error"].mean(),
            -cross_val_rf["test_neg_mean_squared_error"].mean(),
            -cross_val_dt["test_neg_mean_squared_error"].mean(),
            -cross_val_knn["test_neg_mean_squared_error"].mean(),
            -cross_val_en["test_neg_mean_squared_error"].mean(),
        ),
        "RMSE": (
            -cross_val_lr["test_neg_root_mean_squared_error"].mean(),
            -optuna_search_rf.best_score_,
            -optuna_search_dt.best_score_,
            -optuna_search_knn.best_score_,
            -optuna_search_en.best_score_,
        ),
    }
)
scores

In [ ]:
final_estimator = optuna_search_rf.best_estimator_